# C10 Challenger city-swap evaluation

Evaluate challenger models with the same city-swap protocol as the main model block.

This notebook is intentionally split into small cells so partial progress survives crashes.


In [ ]:
import gc
import json
import os
import re
from pathlib import Path
from collections import defaultdict

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


In [ ]:
CWD = Path.cwd().resolve()
NOTEBOOKS_DIR = CWD.parent if CWD.name == 'challengers' else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent

TEST_CSV_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed' / 'test.csv',
    PROJECT_ROOT.parent / 'data' / 'processed' / 'test.csv',
]
LABEL_MAP_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    PROJECT_ROOT / 'label_to_supercategory_v1.csv',
    PROJECT_ROOT.parent / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
]

def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None

TEST_CSV = first_existing(TEST_CSV_CANDIDATES)
LABEL_MAP_CSV = first_existing(LABEL_MAP_CANDIDATES)
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'challengers'
RUNS_CSV = FIGURES_DIR / 'c08_top3_reseed_runs.csv'
FINAL_TABLE_CSV = FIGURES_DIR / 'c09_final_reseed_comparison.csv'
FINAL_DECISION_JSON = FIGURES_DIR / 'c09_final_model_recommendation.json'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'challengers_city_swap'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TEST_CSV:', TEST_CSV)
print('LABEL_MAP_CSV:', LABEL_MAP_CSV)
print('RUNS_CSV exists:', RUNS_CSV.exists())
print('FINAL_DECISION_JSON exists:', FINAL_DECISION_JSON.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)
print('device:', device)

if TEST_CSV is None or LABEL_MAP_CSV is None:
    raise FileNotFoundError('Could not resolve test.csv or label_to_supercategory_v1.csv')


In [ ]:
# Evaluation scope
EVAL_SCOPE = 'scientific_shortlist'  # options: scientific_shortlist, all_reseed_runs, recommended_family, manual
MANUAL_MODEL_DIRS = []
SHORTLIST_FAMILIES = [
    'rdrop_alpha10_2ep',
    'label_smoothing_eps01_2ep',
    'weighted_sampler_citypow15_2ep',
]
SHORTLIST_SINGLE_MODELS = [
    {'model_name': 'bert_gdro_eta005_2ep', 'base_model_name': 'bert_gdro_eta005_2ep', 'method': 'GroupDRO eta=0.05', 'recipe': 'groupdro', 'seed': None, 'model_dir': 'models/challengers/bert_gdro_eta005_2ep'},
    {'model_name': 'focal_gamma1_2ep', 'base_model_name': 'focal_gamma1_2ep', 'method': 'Focal Loss gamma=1.0', 'recipe': 'focal', 'seed': None, 'model_dir': 'notebooks/models/challengers/focal_gamma1_2ep'},
]
SWAP_CITIES = ['Москва', 'Екатеринбург', 'Новосибирск', 'Краснодар', 'Воронеж']
BATCH_SIZE = 8
MAX_LENGTH = 128
SAVE_EVERY_MODEL = True

print('EVAL_SCOPE:', EVAL_SCOPE)
print('SHORTLIST_FAMILIES:', SHORTLIST_FAMILIES)
print('SHORTLIST_SINGLE_MODELS:', SHORTLIST_SINGLE_MODELS)
print('SWAP_CITIES:', SWAP_CITIES)


In [ ]:
runs_df = pd.read_csv(RUNS_CSV) if RUNS_CSV.exists() else pd.DataFrame()
final_df = pd.read_csv(FINAL_TABLE_CSV) if FINAL_TABLE_CSV.exists() else pd.DataFrame()
decision = json.loads(FINAL_DECISION_JSON.read_text()) if FINAL_DECISION_JSON.exists() else {}

print('runs_df rows:', len(runs_df))
print('final_df rows:', len(final_df))
print('decision:', decision)

if not runs_df.empty:
    display(runs_df[['model_name', 'base_model_name', 'method', 'seed', 'accuracy', 'macro_f1', 'worst_gap', 'macro_gap', 'model_dir']])


In [ ]:
if EVAL_SCOPE == 'all_reseed_runs':
    eval_models_df = runs_df.copy()
elif EVAL_SCOPE == 'recommended_family':
    recommended = decision.get('recommended_model')
    eval_models_df = runs_df[runs_df['base_model_name'] == recommended].copy()
elif EVAL_SCOPE == 'scientific_shortlist':
    shortlist_runs_df = runs_df[runs_df['base_model_name'].isin(SHORTLIST_FAMILIES)].copy()
    shortlist_single_df = pd.DataFrame(SHORTLIST_SINGLE_MODELS)
    for col in ['accuracy', 'macro_f1', 'worst_gap', 'macro_gap']:
        if col not in shortlist_single_df.columns:
            shortlist_single_df[col] = np.nan
    eval_models_df = pd.concat([shortlist_runs_df, shortlist_single_df], ignore_index=True, sort=False)
elif EVAL_SCOPE == 'manual':
    rows = []
    for i, rel_path in enumerate(MANUAL_MODEL_DIRS, start=1):
        rows.append({
            'model_name': Path(rel_path).name,
            'base_model_name': Path(rel_path).name,
            'method': 'manual',
            'seed': None,
            'accuracy': np.nan,
            'macro_f1': np.nan,
            'worst_gap': np.nan,
            'macro_gap': np.nan,
            'model_dir': rel_path,
        })
    eval_models_df = pd.DataFrame(rows)
else:
    raise ValueError(f'Unknown EVAL_SCOPE: {EVAL_SCOPE}')

if eval_models_df.empty:
    raise ValueError('No challenger models selected for evaluation')

eval_models_df = eval_models_df.reset_index(drop=True)
eval_models_df.insert(0, 'eval_rank', np.arange(1, len(eval_models_df) + 1))
display_cols = ['eval_rank', 'model_name', 'base_model_name', 'method', 'seed', 'accuracy', 'macro_f1', 'worst_gap', 'macro_gap', 'model_dir']
display(eval_models_df[display_cols])


In [ ]:
CITY_PATTERNS = [
    'санкт-петербург', 'нижний новгород', 'ростов-на-дону',
    'набережные челны', 'магнитогорск', 'новосибирск',
    'екатеринбург', 'красноярск', 'волгоград', 'калининград',
    'владивосток', 'хабаровск', 'ставрополь', 'саратов',
    'челябинск', 'самара', 'казань', 'москва', 'омск',
    'воронеж', 'пермь', 'тюмень', 'томск', 'уфа',
    'тольятти', 'барнаул', 'иркутск', 'пенза', 'липецк',
    'кемерово', 'сочи', 'тверь', 'минск', 'алматы',
    'симферополь', 'ярославль', 'ульяновск', 'ижевск',
    'оренбург', 'мск', 'спб', 'питер',
]
escaped = [re.escape(c) for c in CITY_PATTERNS]
CITY_RE = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE)

CITY_WORDS_SCRUB = [
    'москва', 'московская', 'московский', 'мск',
    'санкт-петербург', 'петербург', 'спб', 'питер', 'ленинград',
    'новосибирск', 'екатеринбург', 'казань', 'нижний новгород',
    'челябинск', 'самара', 'омск', 'ростов-на-дону', 'уфа',
    'красноярск', 'воронеж', 'пермь', 'волгоград',
    'краснодар', 'саратов', 'тюмень', 'тольятти', 'ижевск',
    'барнаул', 'ульяновск', 'иркутск', 'хабаровск', 'ярославль',
    'владивосток', 'махачкала', 'томск', 'оренбург', 'кемерово',
    'новокузнецк', 'рязань', 'астрахань', 'пенза', 'липецк',
    'калининград', 'тула', 'курск', 'ставрополь', 'сочи',
    'магнитогорск', 'томская', 'набережные челны', 'тверь',
    'минск', 'алматы', 'киев', 'симферополь',
    'область', 'край', 'республика', 'регион',
]
AGE_WORDS = [
    'пенсионер', 'пенсионерка', 'пенсия', 'пенсионный',
    'студент', 'студентка', 'выпускник', 'выпускница',
    'молодой', 'молодая', 'junior', 'senior',
]
ALL_SENSITIVE = set(w.lower() for w in CITY_WORDS_SCRUB + AGE_WORDS)

def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ''
    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig and orig[0].isupper() else target_city.lower()
    return CITY_RE.sub(replacer, str(text))

def scrub_text(text, mask_token='[MASK]'):
    if pd.isna(text):
        return ''
    result = str(text)
    for word in sorted(ALL_SENSITIVE, key=len, reverse=True):
        result = re.compile(re.escape(word), re.IGNORECASE).sub(mask_token, result)
    return result

def predict_batch(texts, model, tokenizer, device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
            preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, outputs, preds
    return np.array(all_preds)


In [ ]:
def find_model_files(model_dir):
    model_dir = Path(model_dir)
    if (model_dir / 'config.json').exists():
        return 'hf', model_dir
    if (model_dir / 'bert' / 'config.json').exists() and any((model_dir / name).exists() for name in ['classifier.pt', 'model.pt']):
        return 'split_classifier', model_dir
    candidates = [sub for sub in model_dir.iterdir() if sub.is_dir() and (sub / 'config.json').exists()]
    if candidates:
        preferred = [sub for sub in candidates if sub.name == 'final']
        return 'hf', preferred[0] if preferred else sorted(candidates)[-1]
    return None, None

def _extract_state_dict(obj):
    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'classifier_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return None

def _load_split_classifier_weights(model, model_dir):
    model_dir = Path(model_dir)
    target_state = model.classifier.state_dict()
    for candidate in [model_dir / 'classifier.pt', model_dir / 'model.pt']:
        if not candidate.exists():
            continue
        obj = torch.load(candidate, map_location='cpu')
        state = _extract_state_dict(obj)
        if not isinstance(state, dict):
            continue
        remapped = {}
        for key, value in state.items():
            norm_key = key.replace('module.', '')
            if norm_key.startswith('classifier.'):
                norm_key = norm_key[len('classifier.'):]
            if norm_key in target_state and getattr(value, 'shape', None) == target_state[norm_key].shape:
                remapped[norm_key] = value
        if set(remapped.keys()) == set(target_state.keys()):
            model.classifier.load_state_dict(remapped, strict=True)
            return True
    return False

def load_model_safe(model_dir, device):
    fmt, path = find_model_files(model_dir)
    if fmt is None:
        return None, 'No model files found'
    try:
        if fmt == 'split_classifier':
            bert_path = path / 'bert'
            tokenizer_path = path if (path / 'tokenizer.json').exists() else bert_path
            tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(bert_path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys) and not _load_split_classifier_weights(model, path):
                return None, 'Could not restore split classifier head'
        else:
            tokenizer = AutoTokenizer.from_pretrained(str(path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys):
                return None, f'Missing classifier head: {sorted(missing_keys)}'

        le = None
        for le_path in [path / 'label_encoder.joblib', path.parent / 'label_encoder.joblib', model_dir / 'label_encoder.joblib']:
            if le_path.exists():
                le = joblib.load(le_path)
                break
        if le is None:
            return None, 'No label_encoder.joblib found'

        model = model.to(device).eval()
        return {'model': model, 'tokenizer': tokenizer, 'label_encoder': le}, ''
    except Exception as exc:
        return None, f'{type(exc).__name__}: {exc}'


In [ ]:
df = pd.read_csv(TEST_CSV)
mapping = pd.read_csv(LABEL_MAP_CSV)
label_to_super = dict(zip(mapping['label'], mapping['supercategory']))

df['resume_text'] = df['resume_text'].fillna('').astype(str)
df['text_scrubbed'] = df['resume_text'].apply(scrub_text)

print('Test resumes:', len(df))
print('Columns:', list(df.columns))


In [ ]:
PARTIAL_JSON = OUTPUT_DIR / 'c10_city_swap_partial.json'
RUN_LEVEL_CSV = OUTPUT_DIR / 'c10_city_swap_run_level.csv'
FAMILY_LEVEL_CSV = OUTPUT_DIR / 'c10_city_swap_family_level.csv'
TEXT_REPORT = OUTPUT_DIR / 'c10_city_swap_report.txt'
PLOT_PATH = OUTPUT_DIR / 'c10_city_swap_scatter.png'

if PARTIAL_JSON.exists():
    partial_results = json.loads(PARTIAL_JSON.read_text())
else:
    partial_results = {}

print('Existing partial results:', len(partial_results))
print('Partial file:', PARTIAL_JSON)


In [ ]:
for _, row in eval_models_df.iterrows():
    model_name = row['model_name']
    rel_model_dir = Path(row['model_dir'])
    model_dir = PROJECT_ROOT / rel_model_dir

    print('
' + '=' * 80)
    print('Evaluating:', model_name)
    print('Model dir :', model_dir)

    if model_name in partial_results:
        print('Already present in partial results, skipping')
        continue

    if not model_dir.exists():
        partial_results[model_name] = {
            'model_name': model_name,
            'base_model_name': row.get('base_model_name'),
            'method': row.get('method'),
            'recipe': row.get('recipe'),
            'seed': None if pd.isna(row.get('seed')) else int(row.get('seed')),
            'model_dir': str(rel_model_dir),
            'error': 'directory not found',
        }
        PARTIAL_JSON.write_text(json.dumps(partial_results, indent=2, ensure_ascii=False))
        print('Directory not found')
        continue

    loaded, error = load_model_safe(model_dir, device)
    if loaded is None:
        partial_results[model_name] = {
            'model_name': model_name,
            'base_model_name': row.get('base_model_name'),
            'method': row.get('method'),
            'recipe': row.get('recipe'),
            'seed': None if pd.isna(row.get('seed')) else int(row.get('seed')),
            'model_dir': str(rel_model_dir),
            'error': error,
        }
        PARTIAL_JSON.write_text(json.dumps(partial_results, indent=2, ensure_ascii=False))
        print('Load failed:', error)
        continue

    model = loaded['model']
    tokenizer = loaded['tokenizer']
    le = loaded['label_encoder']

    base_texts = df['resume_text'].tolist()
    orig_preds = predict_batch(base_texts, model, tokenizer, device)

    city_results = {}
    any_flip = np.zeros(len(df), dtype=bool)

    for swap_city in SWAP_CITIES:
        swapped = df['resume_text'].apply(lambda x: swap_cities_in_text(x, swap_city)).tolist()
        swap_preds = predict_batch(swapped, model, tokenizer, device)
        flipped = (swap_preds != orig_preds)
        city_results[swap_city] = {
            'flip_rate': float(flipped.mean()),
            'num_flipped': int(flipped.sum()),
            'num_total': len(df),
        }
        any_flip |= flipped
        print(f'  {swap_city}: {flipped.mean():.3f} ({int(flipped.sum())}/{len(df)})')
        del swapped, swap_preds, flipped
        gc.collect()

    overall_flip = float(any_flip.mean())
    per_class = {}
    class_names = list(le.classes_)
    for i, cls_name in enumerate(class_names):
        mask = orig_preds == i
        if mask.sum() > 0:
            per_class[cls_name] = float(any_flip[mask].mean())

    y_true = le.transform(df['label'].map(label_to_super).fillna('generic_it_ops'))
    acc = float(accuracy_score(y_true, orig_preds))
    f1 = float(f1_score(y_true, orig_preds, average='macro'))

    partial_results[model_name] = {
        'model_name': model_name,
        'base_model_name': row.get('base_model_name'),
        'method': row.get('method'),
        'recipe': row.get('recipe'),
        'seed': None if pd.isna(row.get('seed')) else int(row.get('seed')),
        'model_dir': str(rel_model_dir),
        'accuracy': acc,
        'macro_f1': f1,
        'overall_flip_rate': overall_flip,
        'per_city_flip_rate': city_results,
        'per_class_flip_rate': per_class,
    }

    PARTIAL_JSON.write_text(json.dumps(partial_results, indent=2, ensure_ascii=False))
    print(f'Saved partial result for {model_name}')
    print(f'Acc={acc:.3f} | F1={f1:.3f} | Flip={overall_flip:.3f}')

    del model, tokenizer, le, orig_preds, any_flip
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
rows = []
for model_name, res in partial_results.items():
    row = {
        'model_name': model_name,
        'base_model_name': res.get('base_model_name'),
        'method': res.get('method'),
        'recipe': res.get('recipe'),
        'seed': res.get('seed'),
        'model_dir': res.get('model_dir'),
        'accuracy': res.get('accuracy'),
        'macro_f1': res.get('macro_f1'),
        'overall_flip_rate': res.get('overall_flip_rate'),
        'error': res.get('error', ''),
    }
    for city in SWAP_CITIES:
        row[f'flip_{city}'] = res.get('per_city_flip_rate', {}).get(city, {}).get('flip_rate')
    rows.append(row)

run_level_df = pd.DataFrame(rows).sort_values(['base_model_name', 'seed', 'model_name']).reset_index(drop=True)
run_level_df.to_csv(RUN_LEVEL_CSV, index=False)
run_level_df


In [ ]:
ok_df = run_level_df[run_level_df['error'].fillna('') == ''].copy()
if ok_df.empty:
    family_df = pd.DataFrame()
else:
    family_df = ok_df.groupby(['base_model_name', 'method', 'recipe'], dropna=False).agg(
        runs=('model_name', 'count'),
        seeds=('seed', lambda s: ','.join(str(int(x)) for x in sorted([x for x in s.dropna().tolist()]))),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        flip_mean=('overall_flip_rate', 'mean'),
        flip_std=('overall_flip_rate', 'std'),
    ).reset_index()

    for city in SWAP_CITIES:
        city_mean_df = ok_df.groupby('base_model_name', dropna=False)[f'flip_{city}'].mean().reset_index()
        city_mean_df = city_mean_df.rename(columns={f'flip_{city}': f'flip_{city}_mean'})
        family_df = family_df.merge(city_mean_df, on='base_model_name', how='left')

    family_df = family_df.sort_values(['flip_mean', 'macro_f1_mean'], ascending=[True, False]).reset_index(drop=True)

family_df.to_csv(FAMILY_LEVEL_CSV, index=False)
family_df


In [ ]:
def to_text_report(run_level_df, family_df):
    lines = ['=== Challenger city-swap run-level results ===']
    if run_level_df.empty:
        lines.append('no rows')
    else:
        for _, row in run_level_df.iterrows():
            lines.extend([
                f"model_name: {row['model_name']}",
                f"base_model_name: {row['base_model_name']}",
                f"seed: {row['seed']}",
                f"accuracy: {row['accuracy']}",
                f"macro_f1: {row['macro_f1']}",
                f"overall_flip_rate: {row['overall_flip_rate']}",
                f"error: {row['error'] if row['error'] else '-'}",
                '-' * 60,
            ])

    lines.append('')
    lines.append('=== Challenger city-swap family-level results ===')
    if family_df.empty:
        lines.append('no rows')
    else:
        for _, row in family_df.iterrows():
            lines.extend([
                f"base_model_name: {row['base_model_name']}",
                f"runs: {row['runs']}",
                f"seeds: {row['seeds']}",
                f"accuracy_mean: {row['accuracy_mean']}",
                f"macro_f1_mean: {row['macro_f1_mean']}",
                f"flip_mean: {row['flip_mean']}",
                f"flip_std: {row['flip_std']}",
                'per_city_means: ' + ', '.join(f"{city}={row.get('flip_' + city + '_mean', np.nan)}" for city in SWAP_CITIES),
                '-' * 60,
            ])
    return '\n'.join(lines)

report_text = to_text_report(run_level_df, family_df)
TEXT_REPORT.write_text(report_text)
print(report_text)
print('\nSaved text report to:', TEXT_REPORT)


In [ ]:
if plt is None:
    print('matplotlib not installed; skipping plot')
elif family_df.empty:
    print('No family-level rows to plot')
else:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(
        family_df['flip_mean'],
        family_df['macro_f1_mean'],
        xerr=family_df['flip_std'].fillna(0.0),
        yerr=family_df['macro_f1_std'].fillna(0.0),
        fmt='o',
        capsize=4,
    )
    for _, row in family_df.iterrows():
        ax.annotate(row['base_model_name'], (row['flip_mean'], row['macro_f1_mean']), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('Mean city-swap flip rate (lower is better)')
    ax.set_ylabel('Mean macro-F1 (higher is better)')
    ax.set_title('Challenger city-swap comparison')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved plot to:', PLOT_PATH)
